# Sales

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
%sql
-- drop table  Car_Project.Silver.Sales

In [0]:
# Define Table
silver_table_name = 'Car_Project.Silver.Sales'

# Create table if it doesn't exist
if not spark.catalog.tableExists(silver_table_name):
    spark.sql(f"""
                CREATE TABLE Car_Project.Silver.Sales
                (
                    Model_Key INT NOT NULL,
                    Dealer_Key INT NOT NULL,
                    -- DateKey INT NOT NULL,
                    Revenue DECIMAL(18,2) NOT NULL,
                    Units_Sold INT NOT NULL,
                    CONSTRAINT PK_Fact_Sales PRIMARY KEY (Model_Key, Dealer_Key /*, DateKey*/)
                )
                USING DELTA
              """)
    print("Table created.")
else:
    print("Table already exists.")

In [0]:
df_source = spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .load("abfss://bronze@adlscarproject.dfs.core.windows.net/rawdata")

df_source.display()

In [0]:
# Let's assume df_source has these columns: Model_ID, Dealer_ID, Branch_ID, Date_ID, Revenue, Units_Sold

# Load dimension tables from Delta
df_models = spark.table("Car_Project.Silver.Models").select("Model_Key", "Model_ID")
df_dealers = spark.table("Car_Project.Silver.Dealers").select("Dealer_Key", "Dealer_ID", "Branch_ID")
#df_dates = spark.table("Car_Project.Silver.Dates").select("DateKey", "Date_ID")

# Join df_source with dimensions to get surrogate keys
df_enriched = (df_source
    .join(df_models, on="Model_ID", how="inner")
    .join(df_dealers, on=["Dealer_ID", "Branch_ID"], how="inner")
    #.join(df_dates, on="Date_ID", how="inner")
    .select("Model_Key", "Dealer_Key", "Revenue", "Units_Sold")
)

# Create or replace temp view for MERGE
df_enriched.createOrReplaceTempView("staging_sales")

# Now do the MERGE using SQL syntax
merge_sql = """
MERGE INTO Car_Project.Silver.Sales AS target
USING staging_sales AS source
ON target.Model_Key = source.Model_Key
AND target.Dealer_Key = source.Dealer_Key
WHEN MATCHED THEN
  UPDATE SET target.Revenue = source.Revenue,
             target.Units_Sold = source.Units_Sold
WHEN NOT MATCHED THEN
  INSERT (Model_Key, Dealer_Key, Revenue, Units_Sold)
  VALUES (source.Model_Key, source.Dealer_Key, source.Revenue, source.Units_Sold)
"""

spark.sql(merge_sql)


In [0]:
%sql

Select S.*,M.Model_ID,M.Model_Category,D.Branch_ID,D.Dealer_ID,D.BranchName,D.DealerName
from Car_Project.Silver.Sales S
left join car_project.silver.models M on S.Model_Key=M.Model_Key 
left join car_project.silver.dealers D on D.Dealer_Key=S.Dealer_Key
-- where D.Branch_ID = 'BR2406'

In [0]:
%sql
-- truncate table Car_Project.Silver.Sales